In [3]:
# Topic clustering notebook

from pathlib import Path
import os
from collections import Counter

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

PROJECT_ROOT = Path("..").resolve()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed data dir:", DATA_PROCESSED)


c:\Users\User\Documents\retailmind-prototype\.conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: C:\Users\User\Documents\retailmind-prototype
Processed data dir: C:\Users\User\Documents\retailmind-prototype\data\processed


In [9]:
# Load the issue-tagged dataset produced by issue_tagging.ipynb
tagged_path = DATA_PROCESSED / "uss_english_issue_tagged.parquet"
df = pd.read_parquet(tagged_path)

print("Total rows in tagged dataset:", len(df))
df.head()


Total rows in tagged dataset: 39102


,conv_id,turn_id,speaker,text,dialog_act,scores_raw,scores,is_overall,satisfaction_mean,label,dataset,low_satisfaction,entity_tag,issues,severity,reason
0,0,1,USER,What is the weather like on the March 4th?,INFORM,"3,3,3","[3, 3, 3]",False,3.000000,INFORM,SGD,False,None,[],NONE,
1,0,2,SYSTEM,In which city should I look?,REQUEST,None,None,False,NaN,REQUEST,SGD,False,None,[],NONE,
2,0,3,USER,The weather in Mill Valley.,INFORM,"4,3,3","[4, 3, 3]",False,3.333333,INFORM,SGD,False,None,[],NONE,
3,0,4,SYSTEM,The weather should be around 90 degrees and th...,OFFER,None,None,False,NaN,OFFER,SGD,False,None,[],NONE,
4,0,5,USER,How humid will the temperature be?,REQUEST,"3,3,3","[3, 3, 3]",False,3.000000,REQUEST,SGD,False,None,[],NONE,


In [10]:
import numpy as np

def to_issue_list(val):
    # If already list, keep it
    if isinstance(val, list):
        return val
    # If numpy array, convert to list
    if isinstance(val, np.ndarray):
        return list(val)
    # Anything else (None, float, etc.)
    return []

df["issues"] = df["issues"].apply(to_issue_list)

print("Issues length distribution:")
print(df["issues"].apply(len).value_counts().head())
print("\nFirst 5 normalized issues:")
for i in range(5):
    print(i, "->", df["issues"].iloc[i], " | type:", type(df["issues"].iloc[i]))


Issues length distribution:
issues
0    35013
1     4026
2       63
Name: count, dtype: int64

First 5 normalized issues:
0 -> []  | type: <class 'list'>
1 -> []  | type: <class 'list'>
2 -> []  | type: <class 'list'>
3 -> []  | type: <class 'list'>
4 -> []  | type: <class 'list'>


In [11]:
def has_real_issue(issues):
    # issues is now always a list (after normalization)
    if not isinstance(issues, list) or len(issues) == 0:
        return False
    # If the only label is SUCCESS_BEST_PRACTICE, we don't consider it a failure
    return any(issue != "SUCCESS_BEST_PRACTICE" for issue in issues)

mask_target = (
    (df["low_satisfaction"] == True) &
    (df["speaker"] == "USER") &
    (df["is_overall"] == False) &
    df["issues"].apply(has_real_issue)
)

df_cluster = df[mask_target].copy()
print("Rows selected for clustering:", len(df_cluster))
df_cluster[["dataset", "conv_id", "turn_id", "speaker", "text", "issues", "severity"]].head()


Rows selected for clustering: 3537


,dataset,conv_id,turn_id,speaker,text,issues,severity
10,SGD,0,11,USER,"No, play it on bedroom speaker instead.",[MISSING_CONTEXT],MEDIUM
27,SGD,1,9,USER,No. Don't do that.,[MISSING_CONTEXT],MEDIUM
29,SGD,1,11,USER,Make a reservation at a nearby restaurant.,[UNSUPPORTED_INTENT],HIGH
35,SGD,1,17,USER,"No, I need it for 3 at 11:45 am.",[MISSING_CONTEXT],MEDIUM
52,SGD,2,7,USER,Can you check on another? I'm looking for some...,[MISSING_CONTEXT],MEDIUM


In [14]:
MAX_CLUSTER_ROWS = 5000 
if len(df_cluster) > MAX_CLUSTER_ROWS:
    df_cluster = df_cluster.sample(MAX_CLUSTER_ROWS, random_state=42).copy()
    print("Sampled rows for clustering:", len(df_cluster))


In [16]:
def build_cluster_text(row):
    issues_str = ", ".join(row["issues"]) if isinstance(row["issues"], list) else ""
    reason_str = row.get("reason", "")
    return f"User: {row['text']}\nIssues: {issues_str}\nReason: {reason_str}"

df_cluster["cluster_text"] = df_cluster.apply(build_cluster_text, axis=1)
for i in range(5):
    print("Cluster text example", i)
    print(df_cluster["cluster_text"].iloc[i])
    print()



Cluster text example 0
User: No, play it on bedroom speaker instead.
Issues: MISSING_CONTEXT
Reason: The bot fails to acknowledge the user's request to change the speaker from the kitchen to the bedroom, indicating it did not retain the context of the user's preference.

Cluster text example 1
User: No. Don't do that.
Issues: MISSING_CONTEXT
Reason: The bot fails to acknowledge the user's clear refusal and continues to offer an action that the user has explicitly declined, indicating a lack of understanding of the user's intent.

Cluster text example 2
User: Make a reservation at a nearby restaurant.
Issues: UNSUPPORTED_INTENT
Reason: The bot is unable to fulfill the user's request to make a reservation, indicating a lack of capability to handle such tasks.

Cluster text example 3
User: No, I need it for 3 at 11:45 am.
Issues: MISSING_CONTEXT
Reason: The bot failed to acknowledge the user's previous request for a table, instead asking for confirmation on details that were incorrect. Th

In [17]:
# Load sentence-transformer model (MiniLM)
model = SentenceTransformer("all-MiniLM-L6-v2")

texts = df_cluster["cluster_text"].tolist()
print("Encoding", len(texts), "texts...")

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # optional, helps KMeans sometimes
)

print("Embeddings shape:", embeddings.shape)


c:\Users\User\Documents\retailmind-prototype\.conda\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed

Encoding 3537 texts...


Batches: 100%|██████████| 56/56 [01:03<00:00,  1.13s/it]

Embeddings shape: (3537, 384)


In [18]:
K = 8  # number of topics (to be tuned)

kmeans = KMeans(n_clusters=K, random_state=42, n_init="auto")
topic_ids = kmeans.fit_predict(embeddings)

df_cluster["topic_id"] = topic_ids
df_cluster[["dataset", "conv_id", "turn_id", "text", "issues", "topic_id"]].head()


,dataset,conv_id,turn_id,text,issues,topic_id
10,SGD,0,11,"No, play it on bedroom speaker instead.",[MISSING_CONTEXT],5
27,SGD,1,9,No. Don't do that.,[MISSING_CONTEXT],3
29,SGD,1,11,Make a reservation at a nearby restaurant.,[UNSUPPORTED_INTENT],4
35,SGD,1,17,"No, I need it for 3 at 11:45 am.",[MISSING_CONTEXT],2
52,SGD,2,7,Can you check on another? I'm looking for some...,[MISSING_CONTEXT],7


In [19]:
topics_out_path = DATA_PROCESSED / "uss_english_issue_topics.parquet"
df_cluster.to_parquet(topics_out_path, index=False)
print("Saved per-row topics file to:", topics_out_path)


Saved per-row topics file to: C:\Users\User\Documents\retailmind-prototype\data\processed\uss_english_issue_topics.parquet


In [ ]:
topic_summaries = []

for topic_id, group in df_cluster.groupby("topic_id"):
    n_examples = len(group)

    # Top issues in this topic
    issue_counter = Counter()
    for issues in group["issues"]:
        if isinstance(issues, list):
            issue_counter.update(issues)
    top_issues = [issue for issue, _ in issue_counter.most_common(3)]

    # Example user texts (first 3 rows)
    example_texts = group["text"].head(3).tolist()

    # Optional: one example reason
    example_reason = group["reason"].dropna().astype(str).head(1).tolist()
    example_reason = example_reason[0] if example_reason else ""

    topic_summaries.append({
        "topic_id": topic_id,
        "n_examples": n_examples,
        "top_issues": top_issues,
        "example_texts": example_texts,
        "example_reason": example_reason,
    })

df_topics_summary = pd.DataFrame(topic_summaries).sort_values("n_examples", ascending=False)



,topic_id,n_examples,top_issues,example_texts,example_reason
1,1,1143,"[MISSING_CONTEXT, TONE_ISSUE, UNSUPPORTED_INTENT]","[I want to find a movie to wathc, can you sear...",The bot does not acknowledge the user's intent...
7,7,633,"[MISSING_CONTEXT, UNSUPPORTED_INTENT, HANDOFF_...",[Can you check on another? I'm looking for som...,The bot fails to acknowledge the user's previo...
3,3,578,"[MISSING_CONTEXT, TONE_ISSUE, HANDOFF_REQUIRED]","[No. Don't do that., No I dont want to book an...",The bot fails to acknowledge the user's clear ...
2,2,298,"[MISSING_CONTEXT, WRONG_FACT, UNSUPPORTED_INTENT]","[No, I need it for 3 at 11:45 am., Evening 7:3...",The bot failed to acknowledge the user's previ...
6,6,275,"[MISSING_CONTEXT, UNSUPPORTED_INTENT, WRONG_FACT]","[No, not just yet., No, I want 1 ticket to an ...",The bot fails to acknowledge the user's respon...
4,4,234,"[MISSING_CONTEXT, UNSUPPORTED_INTENT, HANDOFF_...","[Make a reservation at a nearby restaurant., T...",The bot is unable to fulfill the user's reques...
5,5,192,"[MISSING_CONTEXT, UNSUPPORTED_INTENT, WRONG_FACT]","[No, play it on bedroom speaker instead., Bras...",The bot fails to acknowledge the user's reques...
0,0,184,"[MISSING_CONTEXT, UNSUPPORTED_INTENT, HANDOFF_...","[No, I want the appointment at 11:30 on the 5t...",The bot failed to acknowledge the user's reque...


In [23]:
summary_out_path = DATA_PROCESSED / "uss_english_topic_summaries.parquet"
df_topics_summary.to_parquet(summary_out_path, index=False)
print("Saved topic summaries to:", summary_out_path)


Saved topic summaries to: C:\Users\User\Documents\retailmind-prototype\data\processed\uss_english_topic_summaries.parquet


In [24]:
print("Topic distribution:")
df_cluster["topic_id"].value_counts().sort_index()


Topic distribution:


topic_id
0     184
1    1143
2     298
3     578
4     234
5     192
6     275
7     633
Name: count, dtype: int64

In [25]:
def show_topic(topic_id, n_rows=5):
    subset = df_cluster[df_cluster["topic_id"] == topic_id]
    print(f"Topic {topic_id} — {len(subset)} examples")
    print("Top issues:", Counter([iss for lst in subset["issues"] for iss in lst]).most_common(5))
    display(
        subset[["dataset", "conv_id", "turn_id", "text", "issues", "severity", "reason"]]
        .head(n_rows)
    )

for t in sorted(df_cluster["topic_id"].unique()):
    show_topic(t, n_rows=3)


Topic 0 — 184 examples
Top issues: [('MISSING_CONTEXT', 169), ('UNSUPPORTED_INTENT', 15), ('HANDOFF_REQUIRED', 4), ('WRONG_FACT', 2)]


,dataset,conv_id,turn_id,text,issues,severity,reason
290,SGD,10,13,"No, I want the appointment at 11:30 on the 5th.",[MISSING_CONTEXT],HIGH,The bot failed to acknowledge the user's reque...
469,SGD,17,19,"No, it should be the 4th of this month.",[MISSING_CONTEXT],MEDIUM,The bot failed to acknowledge the user's corre...
1669,SGD,63,9,Yes,[MISSING_CONTEXT],MEDIUM,The bot fails to acknowledge that the user has...


Topic 1 — 1143 examples
Top issues: [('MISSING_CONTEXT', 1120), ('TONE_ISSUE', 17), ('UNSUPPORTED_INTENT', 13), ('LOOP', 1)]


,dataset,conv_id,turn_id,text,issues,severity,reason
998,SGD,36,1,I want to find a movie to wathc,[MISSING_CONTEXT],MEDIUM,The bot does not acknowledge the user's intent...
1000,SGD,36,3,can you search for movies shown in San Jose.i ...,"[MISSING_CONTEXT, UNSUPPORTED_INTENT]",MEDIUM,The bot is asking for the user's location desp...
1004,SGD,36,7,I like Documentary movies. could you try again?,[MISSING_CONTEXT],MEDIUM,The bot failed to acknowledge the user's previ...


Topic 2 — 298 examples
Top issues: [('MISSING_CONTEXT', 295), ('WRONG_FACT', 4), ('UNSUPPORTED_INTENT', 2)]


...

,dataset,conv_id,turn_id,text,issues,severity,reason
10,SGD,0,11,"No, play it on bedroom speaker instead.",[MISSING_CONTEXT],MEDIUM,The bot fails to acknowledge the user's reques...
203,SGD,7,3,Brasserie food sounds good.,[UNSUPPORTED_INTENT],MEDIUM,The bot does not recognize 'Brasserie food' as...
207,SGD,7,7,"I want a restaurant that serves alcohol, so an...",[MISSING_CONTEXT],MEDIUM,The bot failed to consider the user's request ...


Topic 6 — 275 examples
Top issues: [('MISSING_CONTEXT', 251), ('UNSUPPORTED_INTENT', 28), ('WRONG_FACT', 3), ('TONE_ISSUE', 1)]


,dataset,conv_id,turn_id,text,issues,severity,reason
66,SGD,2,21,"No, not just yet.",[MISSING_CONTEXT],MEDIUM,The bot fails to acknowledge the user's respon...
190,SGD,6,7,"No, I want 1 ticket to an event on Wednesday n...",[MISSING_CONTEXT],HIGH,The bot failed to acknowledge the user's previ...
194,SGD,6,11,Could you try again? I am looking for tickets ...,[MISSING_CONTEXT],MEDIUM,The bot fails to acknowledge the user's previo...


Topic 7 — 633 examples
Top issues: [('MISSING_CONTEXT', 560), ('UNSUPPORTED_INTENT', 88), ('HANDOFF_REQUIRED', 1), ('LOOP', 1)]


,dataset,conv_id,turn_id,text,issues,severity,reason
52,SGD,2,7,Can you check on another? I'm looking for some...,[MISSING_CONTEXT],MEDIUM,The bot fails to acknowledge the user's previo...
83,SGD,3,3,I need to see a general practitioner.,[UNSUPPORTED_INTENT],MEDIUM,The bot is unable to accommodate the user's re...
205,SGD,7,5,SF please.,[MISSING_CONTEXT],MEDIUM,The bot is asking for the city again despite t...


In [ ]:
'''from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
'''

In [ ]:
'''# OPTIONAL: Use OpenAI text-embedding-3-small instead of MiniLM
# WARNING: This uses your OpenAI quota.

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def get_openai_embeddings(texts, model="text-embedding-3-small", batch_size=128):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = client.embeddings.create(
            model=model,
            input=batch,
        )
        batch_embs = [d.embedding for d in response.data]
        all_embeddings.extend(batch_embs)
    return np.array(all_embeddings, dtype="float32")

# Use the same cluster_texts prepared before
texts = df_cluster["cluster_text"].tolist()
print("Requesting OpenAI embeddings for", len(texts), "texts...")
embeddings_openai = get_openai_embeddings(texts)
print("OpenAI embeddings shape:", embeddings_openai.shape)

# You can then run KMeans again on embeddings_openai and compare topic quality
'''